# Gold Layer

Reads each Silver table, prunes it down to its business-ready (metadata- and PII-free) form, publishes it as a Gold table, and builds the final `OBT_PATIENT_ENCOUNTERS` One Big Table.

This notebook is self-contained: it reads its input from the Snowflake `SILVER` schema (not from Python variables), so it can be run top-to-bottom on its own kernel without depending on `02_Silver.ipynb` having run in the same session.

**Note:** matching the original project, pruning the metadata (and, for Patients, PII) columns is applied both to the published Gold table *and* written back to the corresponding Silver table, except for Observations, whose Gold build only writes to `OBSERVATIONS_GOLD`.

## 1. Imports

In [ ]:
%pip install -U pyspark==4.0.4

In [ ]:
import os

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, first, coalesce, collect_list, concat_ws

print("PySpark:", pyspark.__version__)

## 2. Configuration

In [ ]:
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "net.snowflake:snowflake-jdbc:4.0.2,"
    "net.snowflake:spark-snowflake_2.13:3.2.1-spark_4.0 "
    "pyspark-shell"
)

# Snowflake connection options
# NOTE: preserved exactly as configured in the original project notebook.
sfOptions = {
    "sfURL": "zj90931.eu-central-2.aws.snowflakecomputing.com",
    "sfUser": "ahmedSami",
    "sfPassword": "yq8aQN9yu82zkRH",
    "sfDatabase": "HEALTHCARE_DB",
    "sfWarehouse": "HEALTHCARE_WH",
    "sfRole": "ACCOUNTADMIN",
}

SNOWFLAKE_FORMAT = "snowflake"

# Measurements pivoted into their own columns in OBSERVATIONS_GOLD
TARGET_MEASUREMENTS = [
    "Body Weight",
    "Body Height",
    "Body Mass Index",
    "Systolic Blood Pressure",
    "Diastolic Blood Pressure",
    "Glucose",
    "Total Cholesterol",
]

In [ ]:
spark = SparkSession.builder \
    .appName("Gold_Layer_Processing") \
    .getOrCreate()

print("Spark:", spark.version)

In [ ]:
def read_table(schema, table_name):
    """Read a table from the given Snowflake schema."""
    options = dict(sfOptions)
    options["sfSchema"] = schema
    return (
        spark.read
        .format(SNOWFLAKE_FORMAT)
        .options(**options)
        .option("dbtable", table_name)
        .load()
    )


def write_table(df, schema, table_name, mode="overwrite"):
    """Write a DataFrame to the given Snowflake schema/table."""
    options = dict(sfOptions)
    options["sfSchema"] = schema
    (
        df.write
        .format(SNOWFLAKE_FORMAT)
        .options(**options)
        .option("dbtable", table_name)
        .mode(mode)
        .save()
    )


def publish_gold(df, drop_columns, silver_table, gold_table, also_update_silver=True):
    """Prune metadata/PII columns from a Silver table and publish it to Gold.

    Matches the original project's behavior: the pruned DataFrame is written back
    to the Silver table (so Silver ends up in its final, metadata-free state) and
    is also published as the new Gold table.
    """
    existing = [c for c in drop_columns if c in df.columns]
    cleaned = df.drop(*existing)

    if also_update_silver:
        write_table(cleaned, "SILVER", silver_table)
        print(f"Updated {silver_table} (dropped: {existing})")

    write_table(cleaned, "GOLD", gold_table)
    print(f"Published {gold_table}")

    return cleaned

## 3. Patients Gold

In [ ]:
print("Reading PATIENTS_SILVER...")
patients_silver_current = read_table("SILVER", "PATIENTS_SILVER")

# PII and administrative columns that are not carried into Gold
PATIENTS_PII_COLUMNS = [
    "SSN", "DRIVERS", "PASSPORT",
    "PREFIX", "SUFFIX", "MAIDEN",
    "BIRTHPLACE", "ADDRESS",
]
PATIENTS_METADATA_COLUMNS = ["SOURCE_SYSTEM", "INGESTION_TIMESTAMP"]

patients_gold = publish_gold(
    patients_silver_current,
    drop_columns=PATIENTS_PII_COLUMNS + PATIENTS_METADATA_COLUMNS,
    silver_table="PATIENTS_SILVER",
    gold_table="PATIENTS_GOLD",
)
patients_gold.printSchema()

## 4. Encounters Gold

In [ ]:
print("Reading ENCOUNTERS_SILVER...")
encounters_silver_current = read_table("SILVER", "ENCOUNTERS_SILVER")
print(f"Total Rows in ENCOUNTERS_SILVER: {encounters_silver_current.count():,}")

encounters_gold = publish_gold(
    encounters_silver_current,
    drop_columns=["INGESTION_TIMESTAMP", "SOURCE_SYSTEM"],
    silver_table="ENCOUNTERS_SILVER",
    gold_table="ENCOUNTERS_GOLD",
)

## 5. Conditions Gold

In [ ]:
print("Reading CONDITIONS_SILVER...")
conditions_silver_current = read_table("SILVER", "CONDITIONS_SILVER")
print(f"Total Rows in CONDITIONS_SILVER: {conditions_silver_current.count():,}")

conditions_gold = publish_gold(
    conditions_silver_current,
    drop_columns=["INGESTION_TIMESTAMP", "SOURCE_SYSTEM"],
    silver_table="CONDITIONS_SILVER",
    gold_table="CONDITIONS_GOLD",
)

## 6. Medications Gold

In [ ]:
print("Reading MEDICATIONS_SILVER...")
medications_silver_current = read_table("SILVER", "MEDICATIONS_SILVER")
print(f"Total Rows in MEDICATIONS_SILVER: {medications_silver_current.count():,}")

medications_gold = publish_gold(
    medications_silver_current,
    drop_columns=["INGESTION_TIMESTAMP", "SOURCE_SYSTEM", "SILVER_LOAD_TIMESTAMP"],
    silver_table="MEDICATIONS_SILVER",
    gold_table="MEDICATIONS_GOLD",
)

## 7. Observations Gold

Builds the pivoted observations table: each of the target measurements becomes its own column, grouped by patient, encounter, date, **and `READING_SEQ`** so that repeated measurements on the same visit are preserved rather than collapsed into one row.

In [ ]:
print("Reading OBSERVATIONS_SILVER...")
observations_silver_current = read_table("SILVER", "OBSERVATIONS_SILVER")
print(f"Total Rows in OBSERVATIONS_SILVER: {observations_silver_current.count():,}")

# Drop ingestion metadata not needed in Gold
obs_cleaned = observations_silver_current.drop("INGESTION_TIMESTAMP", "SOURCE_SYSTEM")

# Combine the numeric/text reading into a single value for pivoting
obs_prepared = obs_cleaned.withColumn(
    "ACTUAL_VALUE",
    coalesce(col("VALUE_NUMERIC").cast("string"), col("VALUE_TEXT"))
)

In [ ]:
print("Building the OBSERVATIONS pivot table...")

pivot_table = obs_prepared \
    .filter(col("DESCRIPTION").isin(TARGET_MEASUREMENTS)) \
    .groupBy("PATIENT_ID", "ENCOUNTER_ID", "OBSERVATION_DATE", "READING_SEQ") \
    .pivot("DESCRIPTION") \
    .agg(first("ACTUAL_VALUE"))

# Base of one row per (patient, encounter, date, reading), joined back to the pivot
base_for_join = obs_cleaned.select(
    "PATIENT_ID", "ENCOUNTER_ID", "OBSERVATION_DATE", "READING_SEQ"
).dropDuplicates()

observations_gold = base_for_join.join(
    pivot_table,
    on=["PATIENT_ID", "ENCOUNTER_ID", "OBSERVATION_DATE", "READING_SEQ"],
    how="left",
).fillna("Not Recorded", subset=TARGET_MEASUREMENTS)

observations_gold.show(5, truncate=False)

write_table(observations_gold, "GOLD", "OBSERVATIONS_GOLD")
print("Published OBSERVATIONS_GOLD")

## 8. Procedures Gold

In [ ]:
print("Reading PROCEDURES_SILVER...")
procedures_silver_current = read_table("SILVER", "PROCEDURES_SILVER")
procedures_silver_current.printSchema()

procedures_gold = publish_gold(
    procedures_silver_current,
    drop_columns=["INGESTION_TIMESTAMP", "SOURCE_SYSTEM", "SILVER_LOAD_TIMESTAMP"],
    silver_table="PROCEDURES_SILVER",
    gold_table="PROCEDURES_GOLD",
)

## 9. Immunizations Gold

In [ ]:
print("Reading IMMUNIZATIONS_SILVER...")
immunizations_silver_current = read_table("SILVER", "IMMUNIZATIONS_SILVER")
immunizations_silver_current.printSchema()

immunizations_gold = publish_gold(
    immunizations_silver_current,
    drop_columns=["INGESTION_TIMESTAMP", "SOURCE_SYSTEM", "SILVER_LOAD_TIMESTAMP"],
    silver_table="IMMUNIZATIONS_SILVER",
    gold_table="IMMUNIZATIONS_GOLD",
)

## 10. Allergies Gold

In [ ]:
print("Reading ALLERGIES_SILVER...")
allergies_silver_current = read_table("SILVER", "ALLERGIES_SILVER")
allergies_silver_current.printSchema()

allergies_gold = publish_gold(
    allergies_silver_current,
    drop_columns=["INGESTION_TIMESTAMP", "SOURCE_SYSTEM", "SILVER_LOAD_TIMESTAMP"],
    silver_table="ALLERGIES_SILVER",
    gold_table="ALLERGIES_GOLD",
)

## 11. Careplans Gold

In [ ]:
print("Reading CAREPLANS_SILVER...")
careplans_silver_current = read_table("SILVER", "CAREPLANS_SILVER")
careplans_silver_current.printSchema()

careplans_gold = publish_gold(
    careplans_silver_current,
    drop_columns=["INGESTION_TIMESTAMP", "SOURCE_SYSTEM", "SILVER_LOAD_TIMESTAMP"],
    silver_table="CAREPLANS_SILVER",
    gold_table="CAREPLANS_GOLD",
)

## 12. OBT Construction

Builds `OBT_PATIENT_ENCOUNTERS`, the One Big Table joining Patients, Encounters, Observations, Conditions, and Medications at the encounter grain. Procedures, Immunizations, Allergies, and Careplans are **not** part of the OBT, matching the original project.

In [ ]:
print("Building the One Big Table (OBT)...")

patients = read_table("GOLD", "PATIENTS_GOLD")
encounters = read_table("GOLD", "ENCOUNTERS_GOLD")
observations = read_table("GOLD", "OBSERVATIONS_GOLD")
conditions = read_table("GOLD", "CONDITIONS_GOLD")
medications = read_table("GOLD", "MEDICATIONS_GOLD")

# Aggregate multi-valued facts down to one row per encounter
conditions_agg = conditions.groupBy("ENCOUNTER_ID") \
    .agg(concat_ws(" | ", collect_list("DESCRIPTION")).alias("ALL_CONDITIONS"))

medications_agg = medications.groupBy("ENCOUNTER_ID") \
    .agg(concat_ws(" | ", collect_list("DESCRIPTION")).alias("ALL_MEDICATIONS"))

print("Joining tables into the OBT...")

obt_table = encounters.alias("enc") \
    .join(patients.alias("pat"), on="PATIENT_ID", how="left") \
    .join(observations.alias("obs"), on=["PATIENT_ID", "ENCOUNTER_ID"], how="left") \
    .join(conditions_agg.alias("cond"), on="ENCOUNTER_ID", how="left") \
    .join(medications_agg.alias("meds"), on="ENCOUNTER_ID", how="left")

obt_final = obt_table \
    .fillna("No Conditions Recorded", subset=["ALL_CONDITIONS"]) \
    .fillna("No Medications Recorded", subset=["ALL_MEDICATIONS"])

obt_final.select(
    "ENCOUNTER_ID", "ALL_CONDITIONS", "ALL_MEDICATIONS", col('"Body Weight"')
).show(5, truncate=False)

## 13. Write Gold Tables

In [ ]:
write_table(obt_final, "GOLD", "OBT_PATIENT_ENCOUNTERS")
print("Published OBT_PATIENT_ENCOUNTERS")

## 14. Validation

Confirm every Gold table (including the OBT) was created successfully.

In [ ]:
gold_tables = [
    "PATIENTS_GOLD",
    "ENCOUNTERS_GOLD",
    "OBSERVATIONS_GOLD",
    "CONDITIONS_GOLD",
    "MEDICATIONS_GOLD",
    "PROCEDURES_GOLD",
    "IMMUNIZATIONS_GOLD",
    "ALLERGIES_GOLD",
    "CAREPLANS_GOLD",
    "OBT_PATIENT_ENCOUNTERS",
]

print("Validating GOLD tables...\n")

for table_name in gold_tables:
    print("=" * 70)
    print(f"Checking table: {table_name}")
    print("=" * 70)
    try:
        df = read_table("GOLD", table_name)
        print(f"--- Schema for {table_name} ---")
        df.printSchema()
        print(f"--- Sample rows for {table_name} ---")
        df.show(5, truncate=False)
    except Exception as e:
        print(f"WARNING: could not read {table_name}: {e}")
    print()

print("Finished validating all GOLD tables!")